In [20]:
%load_ext tensorboard
from datasets import load_from_disk
from pathlib import Path
import sys
current_cwd = Path.cwd()
project_root = current_cwd.parent
if str(project_root) not in sys.path :
    sys.path.insert(0, str(project_root))
dataset_path = project_root /"data"/ "sft_dataset"

dataset = load_from_disk(dataset_path)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_id = "Qwen/Qwen2.5-1.5B"
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype = torch.bfloat16, device_map = "auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 162.75it/s]


In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, TaskType

lora_arg = LoraConfig(
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.2,
    bias = "none",
    task_type=TaskType.CAUSAL_LM
)

train_args = SFTConfig(
    output_dir = str(project_root/"outputs"/"sft"),
    per_device_train_batch_size = 2,
    num_train_epochs = 10,
    gradient_accumulation_steps= 4,
    optim = "adamw_torch",
    learning_rate = 2e-5,
    weight_decay = 0.05,
    lr_scheduler_type = "cosine",
    eval_strategy="epoch",
    warmup_steps = 30,
    packing = False,
    logging_steps=1,
    logging_strategy="steps",
    report_to="tensorboard"
    
)
trainer = SFTTrainer(
    model = model,
    args = train_args,
    train_dataset = dataset["train"],
    eval_dataset = dataset["eval"],
    processing_class = tokenizer,
    peft_config=  lora_arg,
)


d:\Anaconda\envs\diffusion\lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
d:\Anaconda\envs\diffusion\lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
## here, you can try to use tensorboard to look your training process while training
%tensorboard --logdir {abs_path} --port 6011

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,2.987739,2.539487
2,2.139549,2.085653
3,1.872297,1.885400
4,0.937936,1.825758
5,0.963643,1.832717
6,1.518931,1.826318
7,1.563134,1.860534
8,1.244226,1.876718
9,0.015361,1.878800
10,1.483008,1.882952


TrainOutput(global_step=310, training_loss=1.3470198031004157, metrics={'train_runtime': 391.2825, 'train_samples_per_second': 6.185, 'train_steps_per_second': 0.792, 'total_flos': 782198626971648.0, 'train_loss': 1.3470198031004157})

In [ ]:
from src.plot_training_metrics import plot_trainer_state_metrics

plot_trainer_state_metrics(
    trainer_state_path=project_root / "outputs" / "sft" / "checkpoint-310" / "trainer_state.json", ## change to your latest model path
    output_dir=project_root / "outputs" / "sft" / "sft_train_metrics",
    run_name="sft_latest",
)


[WindowsPath('c:/Users/Alexander/OneDrive/自学计划/第三周材料/mini-dpo-sft-lab/alignment-mini-lab/outputs/sft/sft_train_metrics/sft_latest_train_mean_token_accuracy.png'),
 WindowsPath('c:/Users/Alexander/OneDrive/自学计划/第三周材料/mini-dpo-sft-lab/alignment-mini-lab/outputs/sft/sft_train_metrics/sft_latest_train_entropy.png'),
 WindowsPath('c:/Users/Alexander/OneDrive/自学计划/第三周材料/mini-dpo-sft-lab/alignment-mini-lab/outputs/sft/sft_train_metrics/sft_latest_train_learning_rate.png'),
 WindowsPath('c:/Users/Alexander/OneDrive/自学计划/第三周材料/mini-dpo-sft-lab/alignment-mini-lab/outputs/sft/sft_train_metrics/sft_latest_train_num_tokens.png'),
 WindowsPath('c:/Users/Alexander/OneDrive/自学计划/第三周材料/mini-dpo-sft-lab/alignment-mini-lab/outputs/sft/sft_train_metrics/sft_latest_train_grad_norm.png'),
 WindowsPath('c:/Users/Alexander/OneDrive/自学计划/第三周材料/mini-dpo-sft-lab/alignment-mini-lab/outputs/sft/sft_train_metrics/sft_latest_train_loss.png'),
 WindowsPath('c:/Users/Alexander/OneDrive/自学计划/第三周材料/mini-dpo-sft-lab/alig